- This code runs 5 independent base queries for seaching 3 languages and the keyword android once in the topic and once in the name, desc and readme
- All 5 queries includes three common qualifier: stars:>50, fork:false and archived:false
- Layer 1: Live Search - GitHub API
- Layer 2: there is no layer two in this version
- This is the final version used to create the list for step 2: AndroidManifest.xml check


In [ ]:
# === Step 1: GitHub URL Lookup with Date-Based Range and Android Detection ===
import os
import requests
import pandas as pd
from dotenv import load_dotenv, set_key
from base64 import b64decode
from datetime import datetime, timedelta
from time import sleep

# === Load tokens and environment ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")
token_index = 0

START_DATE = os.getenv("START_DATE", "2008-01-01")
END_DATE = "2024-12-31"
SEARCH_WINDOW_HOURS = float(os.getenv("SEARCH_WINDOW_HOURS", 24))  # initial window
MIN_WINDOW_HOURS = 0.5
OUTPUT_FILE = "step1_url_lookup_output.csv"

def get_headers():
    global token_index
    headers = {"Authorization": f"token {tokens[token_index]}"}
    token_index = (token_index + 1) % len(tokens)
    return headers

def get_readme(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/readme"
    res = requests.get(url, headers=get_headers())
    if res.status_code == 200:
        content = res.json().get("content", "")
        return b64decode(content).decode("utf-8", errors="ignore")
    return ""

def get_topics(repo_full_name):
    url = f"https://api.github.com/repos/{repo_full_name}/topics"
    res = requests.get(url, headers={**get_headers(), "Accept": "application/vnd.github.mercy-preview+json"})
    if res.status_code == 200:
        return res.json().get("names", [])
    return []

def search_android_keyword(name, desc, topics, readme):
    fields = [name or "", desc or "", " ".join(topics), readme or ""]
    return "yes" if any("android" in f.lower() for f in fields) else "no"

def fetch_metadata(repo):
    full_name = repo.get("full_name")
    url = f"https://api.github.com/repos/{full_name}"
    res = requests.get(url, headers=get_headers())
    if res.status_code == 200:
        repo_data = res.json()
        if repo_data.get("fork") or repo_data.get("archived"):
            return None
        if repo_data.get("stargazers_count", 0) <= 50:
            return None
        lang = repo_data.get("language", "")
        readme = get_readme(full_name)
        topics = get_topics(full_name)
        ak = search_android_keyword(repo_data.get("name"), repo_data.get("description"), topics, readme)
        valid = "yes" if lang in ["Java", "Kotlin", "Dart"] or ak == "yes" else "no"
        return {
            "html_url": repo_data.get("html_url"),
            "name": repo_data.get("name"),
            "language": lang,
            "Android_keyword": ak,
            "Valid_Repo_Step1": valid
        }
    return None

def save_start_date(new_date):
    set_key("All_Tokens.env", "START_DATE", new_date)

def search():
    start = datetime.strptime(START_DATE, "%Y-%m-%d")
    end = datetime.strptime(END_DATE, "%Y-%m-%d")
    window = timedelta(hours=SEARCH_WINDOW_HOURS)

    if os.path.exists(OUTPUT_FILE):
        df_all = pd.read_csv(OUTPUT_FILE)
    else:
        df_all = pd.DataFrame()

    while start < end:
        since = start.strftime("%Y-%m-%dT%H:%M:%SZ")
        until = (start + window).strftime("%Y-%m-%dT%H:%M:%SZ")
        print(f"🔍 Searching from {since} to {until}")

        q = f"created:{since}..{until} stars:>50 language:Java,Kotlin,Dart"
        url = f"https://api.github.com/search/repositories?q={q}&per_page=100&sort=stars&order=desc"
        res = requests.get(url, headers=get_headers())

        if res.status_code == 200:
            items = res.json().get("items", [])
            print(f"✅ Found {len(items)} repos")
            batch = []
            for repo in items:
                meta = fetch_metadata(repo)
                if meta:
                    batch.append(meta)
                sleep(1)
            if batch:
                df_batch = pd.DataFrame(batch)
                df_all = pd.concat([df_all, df_batch], ignore_index=True)
                df_all.drop_duplicates(subset="html_url", inplace=True)
                df_all.to_csv(OUTPUT_FILE, index=False)
            start += window
            save_start_date(start.strftime("%Y-%m-%d"))
        elif res.status_code in [422, 403]:  # Too many results or rate-limited
            if window.total_seconds() / 3600 > MIN_WINDOW_HOURS:
                window /= 2
                print(f"⚠️ Reducing window to {window.total_seconds()/3600:.2f} hours")
            else:
                print("❌ Minimum window reached. Pausing.")
                break
        else:
            print(f"❌ Error: {res.status_code}, {res.text}")
            sleep(10)

    if start >= end:
        print("🎉 Completed full search range. Resetting START_DATE...")
        save_start_date("2008-01-01")

    print("✅ Search complete and output saved.")

if __name__ == "__main__":
    search()


[INFO] 🔎 Remaining: 30 | Resets in 1.0 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-01-01T00:00:00..2008-01-16T00:00:00 → 0 repos
[INFO] 🔎 Remaining: 29 | Resets in 1.0 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 28 | Resets in 1.0 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-01-16T00:00:01..2008-02-15T00:00:01 → 0 repos
[INFO] 🔎 Remaining: 27 | Resets in 1.0 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 26 | Resets in 1.0 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-02-15T00:00:02..2008-04-15T00:00:02 → 0 repos
[INFO] 🔎 Remaining: 25 | Resets in 1.0 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 24 | Resets in 1.0 min
[INFO] ⏳ stars:>50 language:Kotlin fork:false archived:false created:2008-04-15T00:00:03..2008-06-14T00:00:03 → 0 repos
[INFO] 🔎 Remaining: 23 | Resets in 1.0 min
[INFO] ✅ Window done: 0 items
[INFO] 🔎 Remaining: 22 | Resets in 1.0 min
[INFO] ⏳ star

📌 Columns: ['name', 'full_name', 'language', 'stargazers_count', 'forks', 'topics', 'fork', 'archived', 'owner.login', 'html_url', 'clone_url', 'visibility', 'size', 'open_issues_count', 'base_qualifier', 'search_qualifier', 'repo_stars']
                   name                      full_name language  \
0         quran_android            quran/quran_android   Kotlin   
1            gobandroid                ligi/gobandroid   Kotlin   
2               javabot             evanchooly/javabot   Kotlin   
3  facebook-android-sdk  facebook/facebook-android-sdk   Kotlin   
4               tnoodle                 thewca/tnoodle   Kotlin   

   stargazers_count  forks                                             topics  \
0              2174    918                                   [android, quran]   
1               237     66  [android, android-app, goban, kotlin, kotlin-a...   
2                56     30                                                 []   
3              6256   3668        

[INFO] ✅ All done! Clean results saved to: C:\Android Mobile App\Step1_URL_Search\search_results_20250626_193212.csv and C:\Android Mobile App\Step1_URL_Search\search_results_20250626_193212.xlsx


✅ Final rows saved: 108842
